In [53]:
import pandas as pd

chunks_data1 = pd.read_json("data/raw/Books_5.json", lines=True, chunksize=100000)
first_chunk_data1 = next(chunks_data1)

chunks_data2 = pd.read_json("data/raw/Movies_and_TV_5.json", lines=True, chunksize=100000)
first_chunk_data2 = next(chunks_data2)
print(len(first_chunk_data2))

first_chunk_data1.head()


100000


,reviewerID,asin,reviewerName,helpful,reviewText,overall,summary,unixReviewTime,reviewTime
0,A10000012B7CGYKOMPQ4L,000100039X,Adam,"[0, 0]",Spiritually and mentally inspiring! A book tha...,5,Wonderful!,1355616000,"12 16, 2012"
1,A2S166WSCFIFP5,000100039X,"adead_poet@hotmail.com ""adead_poet@hotmail.com""","[0, 2]",This is one my must have books. It is a master...,5,close to god,1071100800,"12 11, 2003"
2,A1BM81XB4QHOA3,000100039X,"Ahoro Blethends ""Seriously""","[0, 0]",This book provides a reflection that you can a...,5,Must Read for Life Afficianados,1390003200,"01 18, 2014"
3,A1MOSTXNIO5MPJ,000100039X,Alan Krug,"[0, 0]",I first read THE PROPHET in college back in th...,5,Timeless for every good and bad time in your l...,1317081600,"09 27, 2011"
4,A2XQ5LZHTD4AFT,000100039X,Alaturka,"[7, 9]",A timeless classic. It is a very demanding an...,5,A Modern Rumi,1033948800,"10 7, 2002"


In [54]:
df_data1 = first_chunk_data1
df_data1["user_id"] = pd.factorize(df_data1["reviewerID"])[0]
df_data1["item_id"] = pd.factorize(df_data1["asin"])[0]
df_data1["interaction"] = 1


df_data2 = first_chunk_data2
df_data2["user_id"] = pd.factorize(df_data2["reviewerID"])[0]
df_data2["item_id"] = pd.factorize(df_data2["asin"])[0]
df_data2["interaction"] = 1



In [55]:
df_data1 = df_data1.sort_values(["user_id", "unixReviewTime"]).copy()

df_test_data1 = df_data1.groupby("user_id").tail(1).copy()
df_train_data1 = df_data1.drop(df_test_data1.index).copy()

df_train_data1 = df_train_data1.reset_index(drop=True)
df_test_data1 = df_test_data1.reset_index(drop=True)



df_data2 = df_data2.sort_values(["user_id", "unixReviewTime"]).copy()

df_test_data2 = df_data2.groupby("user_id").tail(1).copy()
df_train_data2 = df_data2.drop(df_test_data2.index).copy()

df_train_data2 = df_train_data2.reset_index(drop=True)
df_test_data2 = df_test_data2.reset_index(drop=True)


# MostPopular section

In [56]:
item_popularity = (
    df_train_data1.groupby("item_id")
    .size()
    .sort_values(ascending=False)
)

popular_items = item_popularity.index.tolist()
user_seen_train = df_train_data1.groupby("user_id")["item_id"].apply(set).to_dict()

print(popular_items)

[591, 11, 552, 1552, 244, 601, 42, 1873, 1684, 2040, 172, 321, 367, 1310, 315, 758, 814, 391, 2607, 2447, 817, 2135, 1693, 195, 2146, 280, 13, 2351, 593, 691, 757, 292, 2328, 260, 2005, 2011, 386, 274, 2403, 1779, 190, 373, 320, 2038, 2587, 560, 2421, 2169, 2603, 1871, 2103, 457, 136, 1152, 2151, 2520, 1863, 1459, 2393, 1461, 587, 39, 521, 1472, 0, 2064, 1460, 38, 293, 1643, 2614, 1833, 1838, 437, 441, 2375, 1416, 2525, 2524, 1228, 1981, 477, 1205, 1327, 1368, 2523, 276, 2439, 277, 1158, 2522, 159, 1692, 2506, 509, 1988, 1995, 2440, 180, 142, 819, 249, 1581, 2157, 400, 2254, 165, 1058, 2554, 1178, 2012, 1900, 1332, 2325, 219, 833, 2108, 574, 294, 1359, 1296, 364, 575, 325, 1880, 2615, 878, 2019, 415, 2576, 1153, 2359, 1477, 1093, 2102, 553, 446, 455, 635, 1361, 1443, 1832, 1336, 2058, 1805, 745, 2118, 1525, 1388, 1331, 2162, 2109, 2110, 160, 2153, 2507, 2178, 241, 1257, 569, 576, 1656, 69, 1269, 409, 716, 2635, 2424, 1284, 317, 1735, 1734, 1341, 1865, 2211, 1276, 1732, 436, 2390, 2304,

In [57]:
def recommend_most_popular(user_id, popular_items, user_seen_train, k=10):
    seen = user_seen_train.get(user_id, set())
    recs = []
    for item in popular_items:
        if item not in seen:
            recs.append(item)
        if len(recs) == k:
            break
    return recs

def recall_at_k(recommended_items, ground_truth_item):
    return 1.0 if ground_truth_item in recommended_items else 0.0

def precision_at_k(recommended_items, ground_truth_item,k):
    return 1.0/k if ground_truth_item in recommended_items else 0.0

def ndcg_at_k(recommended_items, ground_truth_item):
    if ground_truth_item in recommended_items:
        rank = recommended_items.index(ground_truth_item) + 1
        return 1.0 / math.log2(rank + 1)
    return 0.0


def map(predictions, ground_truth, k=None):
    total_ap = 0
    users = 0

    for user in predictions:
        if user not in ground_truth or len(ground_truth[user]) == 0:
            continue

        ranked = predictions[user][:k] if k else predictions[user]
        relevant = ground_truth[user]

        hits = 0
        sum_precisions = 0

        for i, item in enumerate(ranked, start=1):
            if item in relevant:
                hits += 1
                sum_precisions += hits / i

        total_ap += sum_precisions / len(relevant)
        users += 1

    return total_ap / users if users > 0 else 0


def mrr(predictions, ground_truth, k=None):
    total = 0
    users = 0

    for user in predictions:
        if user not in ground_truth or len(ground_truth[user]) == 0:
            continue

        ranked = predictions[user][:k] if k else predictions[user]
        relevant = ground_truth[user]

        for rank, item in enumerate(ranked, start=1):
            if item in relevant:
                total += 1 / rank
                break

        users += 1

    return total / users if users > 0 else 0

In [58]:
K = 10
recalls = []
ndcgs = []
precisions = []
ground_truths = {}
all_recs = {}

for _, row in df_test_data1.iterrows():
    user_id = row["user_id"]
    ground_truth_item = row["item_id"]

    recs = recommend_most_popular(user_id, popular_items, user_seen_train, k=K)
    ground_truths[user_id] = [ground_truth_item]
    all_recs[user_id] = recs
    recalls.append(recall_at_k(recs, ground_truth_item))
    precisions.append(precision_at_k(recs,ground_truth_item,K))
    ndcgs.append(ndcg_at_k(recs, ground_truth_item))

# PROBABLY NEED TO RE-WRITE THOSE FOR THE REAL THING COZ I REFUSE TO BELIEVE EACH USER WILL ACTUALLY ONLY HAVE ONE GROUND TRUTH ITEM
print(f"Users evaluated: {len(recalls)}")
# metrics to use for ranking specifically: Precision@K and Recall@K
print(f"Precision@{K}: {sum(precisions)/len(precisions):.4f}")
print(f"Recall@{K}: {sum(recalls)/len(recalls):.4f}")
# also nice for within-list ordering checks: NDCG@K, MAP AND MRR
print(f"NDCG@{K}: {sum(ndcgs)/len(ndcgs):.4f}")
print("MAP:", map(all_recs,ground_truths,K))
print("MRR:", mrr(all_recs,ground_truths,K))

Users evaluated: 68136
Precision@10: 0.0200
Recall@10: 0.2004
NDCG@10: 0.1069
MAP: 0.07814970549098915
MRR: 0.07814970549098915


## BRP

In [4]:
import math
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from implicit.bpr import BayesianPersonalizedRanking

# Load Data
chunks_data1 = pd.read_json("data/raw/Books_5.json", lines=True, chunksize=100000)
df_data1 = next(chunks_data1).copy()

df_data1 = df_data1[["reviewerID", "asin", "unixReviewTime"]].copy()
df_data1["interaction"] = 1.0

# Proper factorization
df_data1["user_id"] = pd.factorize(df_data1["reviewerID"])[0]
df_data1["item_id"] = pd.factorize(df_data1["asin"])[0]

# Leave-One-Out Split
df_data1 = df_data1.sort_values(["user_id", "unixReviewTime"]).copy()

df_test_data1 = df_data1.groupby("user_id").tail(1).copy()
df_train_data1 = df_data1.drop(df_test_data1.index).copy()

df_train_data1 = df_train_data1.reset_index(drop=True)
df_test_data1 = df_test_data1.reset_index(drop=True)

# Remove users that lost all train interactions
train_user_counts = df_train_data1["user_id"].value_counts()
valid_users = set(train_user_counts.index)

df_train_data1 = df_train_data1[df_train_data1["user_id"].isin(valid_users)].copy()
df_test_data1 = df_test_data1[df_test_data1["user_id"].isin(valid_users)].copy()

# Remove test items unseen in train
valid_items = set(df_train_data1["item_id"].unique())
df_test_data1 = df_test_data1[df_test_data1["item_id"].isin(valid_items)].copy()

# Remap IDs
user_ids = sorted(df_train_data1["user_id"].unique())
item_ids = sorted(df_train_data1["item_id"].unique())

user_map = {old: new for new, old in enumerate(user_ids)}
item_map = {old: new for new, old in enumerate(item_ids)}

df_train_data1["user_id"] = df_train_data1["user_id"].map(user_map)
df_train_data1["item_id"] = df_train_data1["item_id"].map(item_map)

df_test_data1["user_id"] = df_test_data1["user_id"].map(user_map)
df_test_data1["item_id"] = df_test_data1["item_id"].map(item_map)

df_train_data1 = df_train_data1.dropna().copy()
df_test_data1 = df_test_data1.dropna().copy()

df_train_data1["user_id"] = df_train_data1["user_id"].astype(int)
df_train_data1["item_id"] = df_train_data1["item_id"].astype(int)
df_test_data1["user_id"] = df_test_data1["user_id"].astype(int)
df_test_data1["item_id"] = df_test_data1["item_id"].astype(int)

num_users = int(df_train_data1["user_id"].max()) + 1
num_items = int(df_train_data1["item_id"].max()) + 1

print("Train shape:", df_train_data1.shape)
print("Test shape:", df_test_data1.shape)
print("Users:", num_users, "Items:", num_items)

# Build Sparse Matrix
user_items_csr = csr_matrix(
    (
      df_train_data1["interaction"].astype(np.float32),
      (df_train_data1["user_id"], df_train_data1["item_id"])
    ),
    shape=(num_users, num_items)
)

item_users_csr = user_items_csr.T.tocsr()

model = BayesianPersonalizedRanking(
    factors=64,
    learning_rate=0.05,
    regularization=0.01,
    iterations=100,
    random_state=42
)

model.fit(item_users_csr, show_progress=True)

print("Model user factors:", model.user_factors.shape)
print("Model item factors:", model.item_factors.shape)

# Evaluation
def recall_at_k(recommended_items, ground_truth_item):
    return 1.0 if ground_truth_item in recommended_items else 0.0

def ndcg_at_k(recommended_items, ground_truth_item):
    if ground_truth_item in recommended_items:
        rank = recommended_items.index(ground_truth_item) + 1
        return 1.0 / math.log2(rank + 1)
    return 0.0

K = 10
recalls = []
ndcgs = []

max_user_index = model.user_factors.shape[0] - 1
df_test_data2 = df_test_data1[df_test_data1["user_id"] <= max_user_index].copy()

for _, row in df_test_data2.iterrows():
    user_id = int(row["user_id"])
    ground_truth_item = int(row["item_id"])

    recs, scores = model.recommend(
        userid=user_id,
        user_items=user_items_csr[user_id],
        N=K,
        filter_already_liked_items=True
    )

    recs = recs.tolist()
    recalls.append(recall_at_k(recs, ground_truth_item))
    ndcgs.append(ndcg_at_k(recs, ground_truth_item))

print("\n=== BPR-MF Results ===")
print(f"Users evaluated: {len(recalls)}")
print(f"Recall@{K}: {np.mean(recalls):.5f}")
print(f"NDCG@{K}: {np.mean(ndcgs):.5f}")

Train shape: (31864, 6)
Test shape: (14781, 6)
Users: 14975 Items: 2510


100%|██████████| 100/100 [00:01<00:00, 71.85it/s, train_auc=99.92%, skipped=1.88%]


Model user factors: (2510, 65)
Model item factors: (14975, 65)

=== BPR-MF Results ===
Users evaluated: 2487
Recall@10: 0.00161
NDCG@10: 0.00077


# Lights GCN

In [60]:
import math
import numpy as np
import pandas as pd

from libreco.data import DatasetPure
from libreco.algorithms import LightGCN
from libreco.algorithms import LightGCN


# =========================================================
# 1) Load one chunk per domain
# =========================================================
books_chunk = next(pd.read_json("data/raw/Books_5.json", lines=True, chunksize=100000))
movies_chunk = next(pd.read_json("data/raw/Movies_and_TV_5.json", lines=True, chunksize=100000))

books = books_chunk[["reviewerID", "asin", "unixReviewTime"]].dropna().copy()
movies = movies_chunk[["reviewerID", "asin", "unixReviewTime"]].dropna().copy()

books["domain"] = "Books"
movies["domain"] = "Movies"

print("Books interactions:", len(books))
print("Movies interactions:", len(movies))

Books interactions: 100000
Movies interactions: 100000


In [61]:
# =========================================================
# 2) Keep only shared users
# =========================================================
shared_users = set(books["reviewerID"]).intersection(set(movies["reviewerID"]))

books = books[books["reviewerID"].isin(shared_users)].copy()
movies = movies[movies["reviewerID"].isin(shared_users)].copy()

print("Shared users:", len(shared_users))
print("Books interactions after shared-user filter:", len(books))
print("Movies interactions after shared-user filter:", len(movies))

Shared users: 5008
Books interactions after shared-user filter: 13294
Movies interactions after shared-user filter: 24354


In [62]:
# =========================================================
# 3) Convert to LibRecommender format
#    user is shared across domains
#    item ids must be domain-prefixed so item spaces do not collide
# =========================================================
def to_libreco_format(df):
    out = pd.DataFrame({
        "user": df["reviewerID"].astype(str),
        "item": df["domain"].astype(str) + "::" + df["asin"].astype(str),
        "label": 1.0,
        "time": df["unixReviewTime"].astype(np.int64),
        "domain": df["domain"].astype(str)
    })
    return out

books_df = to_libreco_format(books)
movies_df = to_libreco_format(movies)

print(books_df.head())
print(movies_df.head())

              user               item  label        time domain
1   A2S166WSCFIFP5  Books::000100039X    1.0  1071100800  Books
7   A29TRDMK51GKZR  Books::000100039X    1.0  1383436800  Books
19   A27ZH1AQORJ1L  Books::000100039X    1.0  1066003200  Books
34  A1NPNGWBVD9AK3  Books::000100039X    1.0   961804800  Books
46   AWLFVCT9128JV  Books::000100039X    1.0  1136851200  Books
              user                item  label        time  domain
8    AWF2S3UNW9UA0  Movies::0005019281    1.0  1386201600  Movies
28   AVWWFK3FZDEL2  Movies::0005019281    1.0  1059004800  Movies
64  A1ZIBVOIPBWR3U  Movies::0005019281    1.0  1387497600  Movies
66  A2PSMIRDHYYONP  Movies::0005019281    1.0  1386547200  Movies
71  A17TPT3FWAE5T1  Movies::0005019281    1.0  1008028800  Movies


In [63]:
# =========================================================
# 4) Choose source and target domains
#    Example:
#      source = Books
#      target = Movies
# =========================================================
source_df = books_df.copy()
target_df = movies_df.copy()

SOURCE_NAME = "Books"
TARGET_NAME = "Movies"

In [64]:
# =========================================================
# 5) Filter users so the experiment is feasible
#    Need:
#      - at least 1 source interaction
#      - enough target interactions to do train/test
#
#    For cold-start severity c:
#      keep c target interactions in train
#      last target interaction is held out for test
#
#    So each user needs at least c + 1 target interactions
# =========================================================
def filter_users_for_cold_start(source_df, target_df, cold_k):
    source_counts = source_df["user"].value_counts()
    target_counts = target_df["user"].value_counts()

    valid_users = set(source_counts[source_counts >= 1].index) & \
                  set(target_counts[target_counts >= cold_k + 1].index)

    source_f = source_df[source_df["user"].isin(valid_users)].copy()
    target_f = target_df[target_df["user"].isin(valid_users)].copy()

    return source_f, target_f, valid_users

In [65]:
# =========================================================
# 6) Build target cold-start split
#    For each user in target domain:
#      - last interaction -> test
#      - keep only first cold_k earlier interactions in target train
#
#    This simulates varying cold-start severity in target domain,
#    exactly like your report describes.
# =========================================================
def make_target_cold_start_split(target_df, cold_k):
    target_df = target_df.sort_values(["user", "time"]).copy()

    test_df = target_df.groupby("user").tail(1).copy()
    remaining = target_df.drop(test_df.index).copy()

    # keep earliest cold_k interactions from remaining as target-train
    target_train = (
        remaining.groupby("user", group_keys=False)
        .head(cold_k)
        .copy()
    )

    target_train = target_train.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    return target_train, test_df

In [66]:
# =========================================================
# 7) Build two train sets:
#    A) single-domain baseline = target train only
#    B) cross-domain model = source train + target train
# =========================================================
def build_experiment_frames(source_df, target_df, cold_k):
    source_f, target_f, valid_users = filter_users_for_cold_start(source_df, target_df, cold_k)
    target_train, target_test = make_target_cold_start_split(target_f, cold_k)

    # baseline: target only
    train_single = target_train[["user", "item", "label", "time"]].copy()

    # cross-domain: source + target
    train_cross = pd.concat([
        source_f[["user", "item", "label", "time"]],
        target_train[["user", "item", "label", "time"]]
    ], ignore_index=True)

    test_target = target_test[["user", "item", "label", "time"]].copy()

    return train_single, train_cross, test_target

In [67]:
# =========================================================
# 8) Remove test rows whose target item is unseen in train
#    Needed because the model cannot rank completely unseen items
# =========================================================
def filter_test_seen_in_train(train_df, test_df):
    train_users = set(train_df["user"].unique())
    train_items = set(train_df["item"].unique())

    test_df = test_df[
        test_df["user"].isin(train_users) &
        test_df["item"].isin(train_items)
    ].copy()

    return test_df.reset_index(drop=True)

In [68]:
# =========================================================
# 9) Metric functions
# =========================================================
def recall_at_k(recommended_items, ground_truth_item):
    return 1.0 if ground_truth_item in recommended_items else 0.0

def precision_at_k(recommended_items, ground_truth_item, k):
    return 1.0 / k if ground_truth_item in recommended_items else 0.0

def ndcg_at_k(recommended_items, ground_truth_item):
    if ground_truth_item in recommended_items:
        rank = recommended_items.index(ground_truth_item) + 1
        return 1.0 / math.log2(rank + 1)
    return 0.0

def mean_average_precision(predictions, ground_truth, k=None):
    total_ap = 0.0
    users = 0

    for user in predictions:
        if user not in ground_truth or len(ground_truth[user]) == 0:
            continue

        ranked = predictions[user][:k] if k else predictions[user]
        relevant = ground_truth[user]

        hits = 0
        sum_precisions = 0.0

        for i, item in enumerate(ranked, start=1):
            if item in relevant:
                hits += 1
                sum_precisions += hits / i

        total_ap += sum_precisions / len(relevant)
        users += 1

    return total_ap / users if users > 0 else 0.0

def mean_reciprocal_rank(predictions, ground_truth, k=None):
    total = 0.0
    users = 0

    for user in predictions:
        if user not in ground_truth or len(ground_truth[user]) == 0:
            continue

        ranked = predictions[user][:k] if k else predictions[user]
        relevant = ground_truth[user]

        for rank, item in enumerate(ranked, start=1):
            if item in relevant:
                total += 1.0 / rank
                break

        users += 1

    return total / users if users > 0 else 0.0

In [69]:
# =========================================================
# 10) Train a LibRecommender LightGCN model
# =========================================================
def train_lightgcn_libreco(train_df, n_epochs=20, embed_size=64):
    train_data, data_info = DatasetPure.build_trainset(train_df)

    model = LightGCN(
        task="ranking",
        data_info=data_info,
        loss_type="bpr",
        embed_size=embed_size,
        n_epochs=n_epochs,
        lr=1e-3,
        batch_size=2048,
        num_neg=1,
        device="cuda",
        seed=42,
    )

    model.fit(
        train_data,
        neg_sampling=True,
        verbose=2,
        shuffle=True
    )

    return model, data_info

In [70]:
# =========================================================
# 11) Evaluate on target-domain held-out items
#     Also filter recommendations to TARGET domain only
# =========================================================
def evaluate_target_only(model, test_df, k=10, target_prefix="Movies::"):
    recalls = []
    precisions = []
    ndcgs = []
    all_recs = {}
    ground_truths = {}

    for _, row in test_df.iterrows():
        user = row["user"]
        gt_item = row["item"]

        # Ask for more than k because returned recs may include source-domain items
        rec_dict = model.recommend_user(
            user=user,
            n_rec=100,
            cold_start="popular",
            inner_id=False,
            filter_consumed=True,
            random_rec=False,
        )

        recs = rec_dict[user]
        if isinstance(recs, np.ndarray):
            recs = recs.tolist()
        else:
            recs = list(recs)

        # keep only target-domain items
        recs = [item for item in recs if item.startswith(target_prefix)][:k]

        all_recs[user] = recs
        ground_truths[user] = [gt_item]

        recalls.append(recall_at_k(recs, gt_item))
        precisions.append(precision_at_k(recs, gt_item, k))
        ndcgs.append(ndcg_at_k(recs, gt_item))

    return {
        "users_evaluated": len(recalls),
        f"Precision@{k}": float(np.mean(precisions)) if precisions else 0.0,
        f"Recall@{k}": float(np.mean(recalls)) if recalls else 0.0,
        f"NDCG@{k}": float(np.mean(ndcgs)) if ndcgs else 0.0,
        f"MAP@{k}": mean_average_precision(all_recs, ground_truths, k),
        f"MRR@{k}": mean_reciprocal_rank(all_recs, ground_truths, k),
    }

In [71]:
# =========================================================
# 12) Run one experiment for a given cold-start severity
# =========================================================
def run_one_experiment(source_df, target_df, cold_k, k_eval=10, n_epochs=20):
    print(f"\n========== Cold-start level: keep {cold_k} target train interactions/user ==========")

    train_single, train_cross, test_target = build_experiment_frames(source_df, target_df, cold_k)

    test_single = filter_test_seen_in_train(train_single, test_target)
    test_cross = filter_test_seen_in_train(train_cross, test_target)

    print("Single-domain train size:", len(train_single))
    print("Cross-domain train size:", len(train_cross))
    print("Single-domain test size:", len(test_single))
    print("Cross-domain test size:", len(test_cross))

    model_single, _ = train_lightgcn_libreco(train_single, n_epochs=n_epochs)
    result_single = evaluate_target_only(
        model_single,
        test_single,
        k=k_eval,
        target_prefix=f"{TARGET_NAME}::"
    )

    model_cross, _ = train_lightgcn_libreco(train_cross, n_epochs=n_epochs)
    result_cross = evaluate_target_only(
        model_cross,
        test_cross,
        k=k_eval,
        target_prefix=f"{TARGET_NAME}::"
    )

    return result_single, result_cross

In [72]:
# =========================================================
# 13) Run multiple cold-start severities
#     Example severities: 1, 2, 5
# =========================================================
cold_levels = [1, 2, 5]
results = []

for cold_k in cold_levels:
    single_res, cross_res = run_one_experiment(
        source_df=source_df,
        target_df=target_df,
        cold_k=cold_k,
        k_eval=10,
        n_epochs=20
    )

    row = {
        "cold_k": cold_k,
        "single_users": single_res["users_evaluated"],
        "cross_users": cross_res["users_evaluated"],
        "single_Precision@10": single_res["Precision@10"],
        "cross_Precision@10": cross_res["Precision@10"],
        "single_Recall@10": single_res["Recall@10"],
        "cross_Recall@10": cross_res["Recall@10"],
        "single_NDCG@10": single_res["NDCG@10"],
        "cross_NDCG@10": cross_res["NDCG@10"],
        "single_MAP@10": single_res["MAP@10"],
        "cross_MAP@10": cross_res["MAP@10"],
        "single_MRR@10": single_res["MRR@10"],
        "cross_MRR@10": cross_res["MRR@10"],
    }
    results.append(row)

results_df = pd.DataFrame(results)
print("\n=== Final Results ===")
print(results_df)


========== Cold-start level: keep 1 target train interactions/user ==========
Single-domain train size: 2856
Cross-domain train size: 11800
Single-domain test size: 2529
Cross-domain test size: 2529
Training start time: 2026-03-24 11:42:41


train: 100%|██████████| 2/2 [00:00<00:00, 126.80it/s]


Epoch 1 elapsed: 0.019s
	 train_loss: 0.6193


train: 100%|██████████| 2/2 [00:00<00:00, 134.30it/s]


Epoch 2 elapsed: 0.018s
	 train_loss: 0.6155


...

Epoch 18 elapsed: 0.032s
	 train_loss: 0.6323


train: 100%|██████████| 5/5 [00:00<00:00, 165.62it/s]


Epoch 19 elapsed: 0.032s
	 train_loss: 0.6259


train: 100%|██████████| 5/5 [00:00<00:00, 168.16it/s]


Epoch 20 elapsed: 0.032s
	 train_loss: 0.6188

=== Final Results ===
   cold_k  single_users  cross_users  single_Precision@10  cross_Precision@10  \
0       1          2529         2529             0.002887            0.007394   
1       2          1766         1766             0.002718            0.006965   
2       5           873          873             0.002520            0.005956   

   single_Recall@10  cross_Recall@10  single_NDCG@10  cross_NDCG@10  \
0          0.028865         0.073942        0.014160       0.040135   
1          0.027180         0.069649        0.014215       0.034188   
2          0.025200         0.059565        0.010830       0.028596   

   single_MAP@10  cross_MAP@10  single_MRR@10  cross_MRR@10  
0       0.009791      0.030055       0.009791      0.030055  
1       0.010366      0.023555       0.010366      0.023555  
2       0.006603      0.019311       0.006603      0.019311  


In [73]:
print(train_lightgcn_libreco)

<function train_lightgcn_libreco at 0x7b3100250540>


## Joint BPR

In [3]:
import math
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from implicit.bpr import BayesianPersonalizedRanking

# =========================================================
# 1) Load one chunk per domain
# =========================================================
books_chunk = next(pd.read_json("data/raw/Books_5.json", lines=True, chunksize=100000))
movies_chunk = next(pd.read_json("data/raw/Movies_and_TV_5.json", lines=True, chunksize=100000))

books = books_chunk[["reviewerID", "asin", "unixReviewTime"]].dropna().copy()
movies = movies_chunk[["reviewerID", "asin", "unixReviewTime"]].dropna().copy()

books["domain"] = "Books"
movies["domain"] = "Movies"

print("Books interactions:", len(books))
print("Movies interactions:", len(movies))

# =========================================================
# 2) Keep only shared users
# =========================================================
shared_users = set(books["reviewerID"]).intersection(set(movies["reviewerID"]))

books = books[books["reviewerID"].isin(shared_users)].copy()
movies = movies[movies["reviewerID"].isin(shared_users)].copy()

print("Shared users:", len(shared_users))
print("Books interactions after shared-user filter:", len(books))
print("Movies interactions after shared-user filter:", len(movies))


# =========================================================
# 3) Convert to joint format
#    user ids are shared across domains
#    item ids are domain-prefixed so they do not collide
# =========================================================
def to_joint_format(df):
  out = pd.DataFrame({
    "user": df["reviewerID"].astype(str),
    "item": df["domain"].astype(str) + "::" + df["asin"].astype(str),
    "interaction": 1.0,
    "time": df["unixReviewTime"].astype(np.int64),
    "domain": df["domain"].astype(str)
  })
  return out


books_df = to_joint_format(books)
movies_df = to_joint_format(movies)

print(books_df.head())
print(movies_df.head())

# =========================================================
# 4) Choose source and target domains
# =========================================================
source_df = books_df.copy()
target_df = movies_df.copy()

SOURCE_NAME = "Books"
TARGET_NAME = "Movies"


# =========================================================
# 5) Filter users for cold-start setup
#    Need:
#      - at least 1 source interaction
#      - at least cold_k + 1 target interactions
# =========================================================
def filter_users_for_cold_start(source_df, target_df, cold_k):
  source_counts = source_df["user"].value_counts()
  target_counts = target_df["user"].value_counts()

  valid_users = set(source_counts[source_counts >= 1].index) & \
                set(target_counts[target_counts >= cold_k + 1].index)

  source_f = source_df[source_df["user"].isin(valid_users)].copy()
  target_f = target_df[target_df["user"].isin(valid_users)].copy()

  return source_f, target_f, valid_users


# =========================================================
# 6) Build target cold-start split
#    - last target interaction -> test
#    - first cold_k earlier interactions -> target train
# =========================================================
def make_target_cold_start_split(target_df, cold_k):
  target_df = target_df.sort_values(["user", "time"]).copy()

  test_df = target_df.groupby("user").tail(1).copy()
  remaining = target_df.drop(test_df.index).copy()

  target_train = (
    remaining.groupby("user", group_keys=False)
    .head(cold_k)
    .copy()
  )

  target_train = target_train.reset_index(drop=True)
  test_df = test_df.reset_index(drop=True)

  return target_train, test_df


# =========================================================
# 7) Build joint train/test
#    joint train = all source interactions + limited target train
# =========================================================
def build_joint_bpr_frames(source_df, target_df, cold_k):
  source_f, target_f, valid_users = filter_users_for_cold_start(source_df, target_df, cold_k)
  target_train, target_test = make_target_cold_start_split(target_f, cold_k)

  train_joint = pd.concat([
    source_f[["user", "item", "interaction", "time"]],
    target_train[["user", "item", "interaction", "time"]]
  ], ignore_index=True)

  test_target = target_test[["user", "item", "interaction", "time"]].copy()

  return train_joint, test_target


# =========================================================
# 8) Remove test rows whose user/item is unseen in train
# =========================================================
def filter_test_seen_in_train(train_df, test_df):
  train_users = set(train_df["user"].unique())
  train_items = set(train_df["item"].unique())

  test_df = test_df[
    test_df["user"].isin(train_users) &
    test_df["item"].isin(train_items)
    ].copy()

  return test_df.reset_index(drop=True)


# =========================================================
# 9) Metrics
# =========================================================
def recall_at_k(recommended_items, ground_truth_item):
  return 1.0 if ground_truth_item in recommended_items else 0.0


def precision_at_k(recommended_items, ground_truth_item, k):
  return 1.0 / k if ground_truth_item in recommended_items else 0.0


def ndcg_at_k(recommended_items, ground_truth_item):
  if ground_truth_item in recommended_items:
    rank = recommended_items.index(ground_truth_item) + 1
    return 1.0 / math.log2(rank + 1)
  return 0.0


def mean_average_precision(predictions, ground_truth, k=None):
  total_ap = 0.0
  users = 0

  for user in predictions:
    if user not in ground_truth or len(ground_truth[user]) == 0:
      continue

    ranked = predictions[user][:k] if k else predictions[user]
    relevant = ground_truth[user]

    hits = 0
    sum_precisions = 0.0

    for i, item in enumerate(ranked, start=1):
      if item in relevant:
        hits += 1
        sum_precisions += hits / i

    total_ap += sum_precisions / len(relevant)
    users += 1

  return total_ap / users if users > 0 else 0.0


def mean_reciprocal_rank(predictions, ground_truth, k=None):
  total = 0.0
  users = 0

  for user in predictions:
    if user not in ground_truth or len(ground_truth[user]) == 0:
      continue

    ranked = predictions[user][:k] if k else predictions[user]
    relevant = ground_truth[user]

    for rank, item in enumerate(ranked, start=1):
      if item in relevant:
        total += 1.0 / rank
        break

    users += 1

  return total / users if users > 0 else 0.0


# =========================================================
# 10) Build sparse matrix with tight remapping
# =========================================================
def build_joint_sparse_matrix(train_df):
  train_df = train_df.copy()

  user_ids = sorted(train_df["user"].unique())
  item_ids = sorted(train_df["item"].unique())

  user2id = {u: i for i, u in enumerate(user_ids)}
  item2id = {it: i for i, it in enumerate(item_ids)}
  id2item = {i: it for it, i in item2id.items()}

  train_df["user_id"] = train_df["user"].map(user2id).astype(int)
  train_df["item_id"] = train_df["item"].map(item2id).astype(int)

  num_users = int(train_df["user_id"].max()) + 1
  num_items = int(train_df["item_id"].max()) + 1

  print("Users:", num_users, "Items:", num_items)

  user_items_csr = csr_matrix(
    (
      train_df["interaction"].astype(np.float32),
      (train_df["user_id"], train_df["item_id"])
    ),
    shape=(num_users, num_items)
  )

  return train_df, user_items_csr, user2id, item2id, id2item


# =========================================================
# 11) Train joint BPR
#    IMPORTANT:
#    implicit BPR should be fit on USER-ITEM matrix here
# =========================================================
def train_joint_bpr(train_df, factors=64, lr=0.05, reg=0.01, iterations=100):
  train_mapped, user_items_csr, user2id, item2id, id2item = build_joint_sparse_matrix(train_df)

  model = BayesianPersonalizedRanking(
    factors=factors,
    learning_rate=lr,
    regularization=reg,
    iterations=iterations,
    random_state=42
  )

  model.fit(user_items_csr, show_progress=True)

  print("Model user factors:", model.user_factors.shape)
  print("Model item factors:", model.item_factors.shape)

  return model, user_items_csr, user2id, item2id, id2item


# =========================================================
# 12) Evaluate only on target-domain items
#    same idea as your LightGCN evaluation:
#    recommend broadly, then filter to target prefix
# =========================================================
def evaluate_joint_bpr_target_only(model, user_items_csr, user2id, item2id, id2item, test_df, k=10,
                                   target_prefix="Movies::"):
  recalls = []
  precisions = []
  ndcgs = []
  all_recs = {}
  ground_truths = {}

  max_user_index = user_items_csr.shape[0] - 1
  num_items = user_items_csr.shape[1]

  for _, row in test_df.iterrows():
    user = row["user"]
    gt_item = row["item"]

    if user not in user2id:
      continue
    if gt_item not in item2id:
      continue

    user_id = int(user2id[user])

    if user_id < 0 or user_id > max_user_index:
      continue

    # ask for more than k because many recs may belong to the source domain
    n_candidates = min(200, num_items)

    rec_ids, scores = model.recommend(
      userid=user_id,
      user_items=user_items_csr[user_id],
      N=n_candidates,
      filter_already_liked_items=True
    )

    if isinstance(rec_ids, np.ndarray):
      rec_ids = rec_ids.tolist()
    else:
      rec_ids = list(rec_ids)

    rec_items = [id2item[i] for i in rec_ids]
    rec_items = [item for item in rec_items if item.startswith(target_prefix)][:k]

    all_recs[user] = rec_items
    ground_truths[user] = [gt_item]

    recalls.append(recall_at_k(rec_items, gt_item))
    precisions.append(precision_at_k(rec_items, gt_item, k))
    ndcgs.append(ndcg_at_k(rec_items, gt_item))

  return {
    "users_evaluated": len(recalls),
    f"Precision@{k}": float(np.mean(precisions)) if precisions else 0.0,
    f"Recall@{k}": float(np.mean(recalls)) if recalls else 0.0,
    f"NDCG@{k}": float(np.mean(ndcgs)) if ndcgs else 0.0,
    f"MAP@{k}": mean_average_precision(all_recs, ground_truths, k),
    f"MRR@{k}": mean_reciprocal_rank(all_recs, ground_truths, k),
  }


# =========================================================
# 13) Run one experiment
# =========================================================
def run_joint_bpr_experiment(source_df, target_df, cold_k, k_eval=10, factors=64, iterations=100):
  print(f"\n========== Joint BPR cold-start level: keep {cold_k} target train interactions/user ==========")

  train_joint, test_target = build_joint_bpr_frames(source_df, target_df, cold_k)
  test_target = filter_test_seen_in_train(train_joint, test_target)

  print("Joint train size:", len(train_joint))
  print("Joint test size:", len(test_target))

  model, user_items_csr, user2id, item2id, id2item = train_joint_bpr(
    train_joint,
    factors=factors,
    lr=0.05,
    reg=0.01,
    iterations=iterations
  )

  valid_users = set(user2id.keys())
  valid_items = set(item2id.keys())

  test_target = test_target[
    test_target["user"].isin(valid_users) &
    test_target["item"].isin(valid_items)
    ].copy()

  results = evaluate_joint_bpr_target_only(
    model=model,
    user_items_csr=user_items_csr,
    user2id=user2id,
    item2id=item2id,
    id2item=id2item,
    test_df=test_target,
    k=k_eval,
    target_prefix=f"{TARGET_NAME}::"
  )

  return results


# =========================================================
# 14) Run multiple cold-start severities
# =========================================================
cold_levels = [1, 2, 5]
results = []

for cold_k in cold_levels:
  joint_res = run_joint_bpr_experiment(
    source_df=source_df,
    target_df=target_df,
    cold_k=cold_k,
    k_eval=10,
    factors=64,
    iterations=100
  )

  row = {
    "cold_k": cold_k,
    "joint_users": joint_res["users_evaluated"],
    "joint_Precision@10": joint_res["Precision@10"],
    "joint_Recall@10": joint_res["Recall@10"],
    "joint_NDCG@10": joint_res["NDCG@10"],
    "joint_MAP@10": joint_res["MAP@10"],
    "joint_MRR@10": joint_res["MRR@10"],
  }
  results.append(row)

results_df = pd.DataFrame(results)
print("\n=== Joint BPR-MF Results ===")
print(results_df)

Books interactions: 100000
Movies interactions: 100000
Shared users: 5008
Books interactions after shared-user filter: 13294
Movies interactions after shared-user filter: 24354
              user               item  interaction        time domain
1   A2S166WSCFIFP5  Books::000100039X          1.0  1071100800  Books
7   A29TRDMK51GKZR  Books::000100039X          1.0  1383436800  Books
19   A27ZH1AQORJ1L  Books::000100039X          1.0  1066003200  Books
34  A1NPNGWBVD9AK3  Books::000100039X          1.0   961804800  Books
46   AWLFVCT9128JV  Books::000100039X          1.0  1136851200  Books
              user                item  interaction        time  domain
8    AWF2S3UNW9UA0  Movies::0005019281          1.0  1386201600  Movies
28   AVWWFK3FZDEL2  Movies::0005019281          1.0  1059004800  Movies
64  A1ZIBVOIPBWR3U  Movies::0005019281          1.0  1387497600  Movies
66  A2PSMIRDHYYONP  Movies::0005019281          1.0  1386547200  Movies
71  A17TPT3FWAE5T1  Movies::0005019281     

100%|██████████| 100/100 [00:00<00:00, 162.90it/s, train_auc=99.92%, skipped=1.46%]


Model user factors: (2856, 65)
Model item factors: (2673, 65)

========== Joint BPR cold-start level: keep 2 target train interactions/user ==========
Joint train size: 10868
Joint test size: 1766
Users: 1917 Items: 2595


100%|██████████| 100/100 [00:00<00:00, 152.86it/s, train_auc=99.97%, skipped=1.65%]


Model user factors: (1917, 65)
Model item factors: (2595, 65)

========== Joint BPR cold-start level: keep 5 target train interactions/user ==========
Joint train size: 8930
Joint test size: 873
Users: 929 Items: 2297


100%|██████████| 100/100 [00:00<00:00, 251.55it/s, train_auc=100.00%, skipped=1.75%]


Model user factors: (929, 65)
Model item factors: (2297, 65)

=== Joint BPR-MF Results ===
   cold_k  joint_users  joint_Precision@10  joint_Recall@10  joint_NDCG@10  \
0       1         2529            0.002017         0.020166       0.009843   
1       2         1766            0.001982         0.019819       0.009660   
2       5          873            0.002864         0.028637       0.012267   

   joint_MAP@10  joint_MRR@10  
0      0.006791      0.006791  
1      0.006529      0.006529  
2      0.007416      0.007416  
